# Fake News Detection — Task 7: AI-Powered Verification & Cross-Source Analysis

---

## System Architecture

This notebook implements **Step 7: AI-Powered News Verification & Cross-Source Analysis**.

Step 7 adds an intelligent investigation and cross-source verification layer on top of our existing ML model.

```
                         NEWS ARTICLE
                              │
                    ┌─────────────────┐
                    │ Step 5: ML      │
                    │ Prediction      │
                    └────────┬────────┘
                             │ FAKE / REAL + Confidence %
                             ▼
                    ┌─────────────────┐
                    │ Step 6:         │
                    │ Explainability  │
                    └────────┬────────┘
                             │ Top Features + Clickbait Signals
                             ▼
                    ┌─────────────────┐
                    │ Step 7: AI      │
                    │ Verification    │
                    └────────┬────────┘
                             │
        ┌────────────────────┼────────────────────┐
        ▼                    ▼                    ▼
  Extract Claims       Web & News Search    Official & Social
  (3-7 assertions)     (DuckDuckGo API)     (.gov / .org / X)
        └────────────────────┬────────────────────┘
                             │
                             ▼
                  Source Tier Classification
                  (Tier 1, Tier 2, Tier 3)
                             │
                             ▼
                    Cross-Source Compare
                    & Independence Check
                             │
                             ▼
                    FINAL VERIFICATION REPORT
                    (ML vs AI Comparison)
```

---

### Core Principles
1. **Do NOT retrain or replace the ML model**: Steps 1-6 remain byte-for-byte untouched.
2. **Evidence-Based Reasoning**: The AI checks external sources rather than stating opinions as fact.
3. **Source Priority**: Categorizes sources into **Tier 1 (Official/Primary)**, **Tier 2 (Established News)**, and **Tier 3 (Other)**.
4. **Source Independence**: Detects when multiple outlets merely syndicate or copy the same wire report.
5. **ML vs AI Comparison**: Explicitly flags when ML statistical classification and external AI evidence analysis disagree without silently overriding ML.
6. **API Security**: Uses environment variables (`.env`) for keys. Excludes `.env` via `.gitignore`.
7. **Graceful Fallback**: If LLM/Search APIs fail or keys are missing, the ML prediction remains fully operational.

---

## Section 1: Setup & Environment Configuration

We load `.env` to configure optional LLM API keys securely without exposing them in code.

In [ ]:
import sys
import os
import json
import warnings
warnings.filterwarnings('ignore')

# Add src/ to Python module path
sys.path.insert(0, '../src')

from dotenv import load_dotenv
load_dotenv('../.env')

# Import project modules
import prediction as pred
import explainability as exp
import web_search as ws
import source_analysis as sa
import ai_verification as av

print('All modules successfully loaded!')
print('ML Model Loaded     :', type(pred.model).__name__)
print('Vectorizer Vocab     :', f'{len(pred.vectorizer.vocabulary_):,} features')
print('Gemini API Key Set  :', bool(os.environ.get('GEMINI_API_KEY')))

---

## Section 2: Verification of Existing Pipeline Integration (Steps 5 & 6)

Step 7 reuses `src/prediction.py` and `src/explainability.py` directly.

In [ ]:
sample_text = (
    'WASHINGTON (Reuters) - The Federal Reserve on Wednesday raised its benchmark '
    'interest rate by a quarter percentage point, citing continued strength in the '
    'labor market and persistent inflation pressures. Fed Chair Jerome Powell said '
    'the central bank remains committed to its two percent inflation target over the '
    'medium term, following the two-day policy meeting held in Washington.'
)

ml_res = pred.predict_news(sample_text)
exp_res = exp.get_explanation(sample_text, top_n=5)

print('=== STEP 5 ML PREDICTION ===')
print('Prediction :', ml_res['prediction'])
print('Confidence :', f"{ml_res['confidence']}%")
print()
print('=== STEP 6 EXPLAINABILITY ===')
print('Top Features :', [f['word'] for f in exp_res['influential_features']])
print('Suspicious   :', exp_res['suspicious_language'])

---

## Section 3: Factual Claim Extraction Module

Extracts 3-7 verifiable factual claims from an article, filtering out opinions or subjective statements.

In [ ]:
extracted_claims = av.extract_claims(sample_text, max_claims=5)

print(f'Extracted {len(extracted_claims)} factual claims:')
for i, claim in enumerate(extracted_claims, 1):
    print(f'  {i}. [{claim["importance"].upper()}] Verifiable: {claim["verifiable"]}')
    print(f'     "{claim["claim"]}"')

---

## Section 4: Live Web Search & Source Priority Classification

Searches DuckDuckGo for general reporting, official primary sources (`.gov`, `.org`), and social media accounts.
Classifies each retrieved source into:
- **Tier 1 — Official / Primary Source**: Government, official organizations, press releases.
- **Tier 2 — Established News Organization**: Reuters, BBC, AP, WSJ, The Hindu, etc.
- **Tier 3 — Other / Secondary**: General web, blogs, unverified outlets.

In [ ]:
claim_statement = extracted_claims[0]['claim']

web_results = ws.search_claim(claim_statement, max_results=3)
official_results = ws.find_official_sources(claim_statement, max_results=2)
social_results = ws.find_social_confirmation(claim_statement, max_results=2)

all_sources = web_results + official_results
analyzed = sa.analyze_sources(all_sources)

print('=== SOURCE TIER DISTRIBUTION ===')
print('Total Sources Analyzed :', analyzed['total_sources'])
print('Tier 1 (Official)      :', analyzed['tier1_count'])
print('Tier 2 (Established)   :', analyzed['tier2_count'])
print('Tier 3 (Other)         :', analyzed['tier3_count'])
print()
print('=== RETRIEVED SOURCES DETAILS ===')
for src in analyzed['classified_sources'][:4]:
    print(f"[{src['tier_name']}] {src['source']} ({src['credibility_level']})")
    print(f"  Title: {src['title']}")
    print(f"  URL  : {src['url']}")
    print()

---

## Section 5: Source Independence & Syndication Analysis

Checks if multiple news outlets are reporting independently or simply syndicating the same wire report.

In [ ]:
indep_analysis = sa.detect_source_independence(all_sources)

print('Independence Status :', indep_analysis['independence'])
print('Primary Wire        :', indep_analysis['primary_wire_detected'])
print('Detailed Analysis   :', indep_analysis['analysis'])

---

## Section 6: Comprehensive Verification Test Cases (Tests A - E)

We evaluate the complete AI Verification Engine (`verify_article`) across 5 realistic test cases:

| Test | Scenario | Expected ML / AI Outcome |
|---|---|---|
| **Test A** | Real news article with strong external reporting | REAL / HIGHLY_SUPPORTED |
| **Test B** | Questionable / sensational claim | FAKE / CONFLICTING or UNVERIFIED |
| **Test C** | Claim with conflicting reporting | PARTIALLY_SUPPORTED or CONFLICTING |
| **Test D** | Unverified claim with no external online reporting | UNVERIFIED |
| **Test E** | Article involving entity with official social accounts | Official Social Accounts Discovered |

In [ ]:
# ── TEST A: Formal news story with strong external confirmation ──
test_a = (
    'WASHINGTON (Reuters) - The Federal Reserve on Wednesday raised its benchmark '
    'interest rate by a quarter percentage point for the fourth time this year, '
    'citing continued strength in the labor market and persistent inflation pressures. '
    'Fed Chair Jerome Powell said in a statement that the central bank remains '
    'committed to bringing inflation back to its two percent target over the medium term.'
)

print('Running Test A ...')
report_a = av.verify_article(test_a)
av.display_verification_report(report_a)

In [ ]:
# ── TEST B: Questionable / sensational clickbait claim ──
test_b = (
    'BOMBSHELL REVELATION! Scientists HATE this one secret! Big Pharma has been hiding '
    'a 100% guaranteed miracle cure for all diseases for years. Share this now before '
    'the government deletes this video! MAKE THIS VIRAL!'
)

print('Running Test B ...')
report_b = av.verify_article(test_b)
av.display_verification_report(report_b)

In [ ]:
# ── TEST C: Claim with conflicting reporting ──
test_c = (
    'Global Tech Company X announced a massive secret merger with Competitor Y today. '
    'However, regulatory officials denied receiving any filing, while rival news outlets '
    'report that merger talks completely collapsed last week.'
)

print('Running Test C ...')
report_c = av.verify_article(test_c)
av.display_verification_report(report_c)

In [ ]:
# ── TEST D: Unverified claim where no reliable confirmation exists ──
test_d = (
    'A small local shop in Anytown XYZ allegedly discovered a 500-year-old mysterious artifact '
    'underneath their wooden floorboards yesterday afternoon during routine plumbing repairs.'
)

print('Running Test D ...')
report_d = av.verify_article(test_d)
av.display_verification_report(report_d)

In [ ]:
# ── TEST E: Entity with official public social media account ──
test_e = (
    'The Federal Reserve (@federalreserve) released an official statement on monetary policy '
    'adjustments and inflation target projections for the upcoming fiscal quarter.'
)

print('Running Test E ...')
report_e = av.verify_article(test_e)
av.display_verification_report(report_e)

---

## Section 7: ML vs AI Comparison & Disagreement Handling

Demonstrates how the system handles cases where the ML model prediction and external AI evidence analysis disagree.

In [ ]:
# Construct a case where ML style prediction and External Evidence might disagree
disagreement_text = (
    'WASHINGTON (Reuters) - The Federal Reserve announced an unexpected emergency interest rate cut today. '
    'Official government records confirm the decision, but social media commentary claims the opposite.'
)

disagreement_report = av.verify_article(disagreement_text)

print('ML Prediction :', disagreement_report['ml_classification']['prediction'])
print('AI Status     :', disagreement_report['verification_summary']['overall_status'])
print('Disagreement  :', disagreement_report['verification_summary']['disagreement_detected'])
if disagreement_report['verification_summary']['disagreement_detected']:
    print('Warning Msg   :', disagreement_report['verification_summary']['disagreement_warning'])

---

## Section 8: Error Resilience & Graceful Fallback

Demonstrates that if an external API key is missing or search encounters an error, the system does NOT crash.
The ML model and Step 6 explainability remain 100% operational.

In [ ]:
# Simulate empty input validation
err_report = av.verify_article('')
print('Empty Input Error Handling :', err_report['error'])
assert 'error' in err_report

# Verify ML resilience even when external search returns empty
fallback_search = ws.search_claim('nonexistent_random_gibberish_term_123456')
print('Fallback Search Count      :', len(fallback_search))
assert len(fallback_search) >= 0

---

## Section 9: Final Requirements Verification

Runs 10 comprehensive assertion checks verifying all Step 7 requirements.

In [ ]:
print('=' * 60)
print('  STEP 7: FINAL VERIFICATION CHECKS')
print('=' * 60)
print()

# 1. Existing ML model works
assert pred.model is not None, 'ML Model not loaded!'
print('  [OK] 1. Existing ML model loads successfully.')

# 2. Step 5 prediction pipeline works
p_res = pred.predict_news(sample_text)
assert 'prediction' in p_res and 'confidence' in p_res, 'Step 5 failed!'
print('  [OK] 2. Step 5 prediction pipeline works.')

# 3. Step 6 explainability works
e_res = exp.get_explanation(sample_text)
assert 'influential_features' in e_res, 'Step 6 failed!'
print('  [OK] 3. Step 6 explainability works.')

# 4. Claim extraction works
claims_test = av.extract_claims(sample_text)
assert len(claims_test) > 0, 'Claim extraction failed!'
print(f'  [OK] 4. Claims extracted successfully ({len(claims_test)} claims).')

# 5. Web search works
search_test = ws.search_claim(claims_test[0]['claim'])
assert isinstance(search_test, list), 'Search failed!'
print(f'  [OK] 5. Web search returns valid structured results ({len(search_test)} sources).')

# 6. Source tier classification works
t_test = sa.classify_source_tier('https://www.reuters.com')
assert t_test['tier'] == 'TIER_2_ESTABLISHED_NEWS', 'Tier classification failed!'
print('  [OK] 6. Source tier classification correctly identifies Tier 1/2/3.')

# 7. Public social media check works
soc_test = ws.find_social_confirmation(claims_test[0]['claim'])
assert 'status' in soc_test, 'Social check failed!'
print('  [OK] 7. Public social media account confirmation check works.')

# 8. Evidence comparison & status generation works
v_report = av.verify_article(sample_text)
assert 'verification_summary' in v_report, 'Verification report failed!'
assert v_report['verification_summary']['overall_status'] in ['HIGHLY_SUPPORTED', 'PARTIALLY_SUPPORTED', 'CONFLICTING_EVIDENCE', 'UNVERIFIED']
print('  [OK] 8. Cross-source evidence comparison & overall status generated.')

# 9. ML vs AI comparison works
assert 'disagreement_detected' in v_report['verification_summary']
print('  [OK] 9. ML vs AI comparison & disagreement detection operational.')

# 10. API security & error handling works
assert os.path.exists('../.gitignore'), '.gitignore missing!'
with open('../.gitignore') as f: git_content = f.read()
assert '.env' in git_content, '.env not in .gitignore!'
print('  [OK] 10. API key security verified (.env is in .gitignore).')

print()
print('All 10 verification assertions PASSED!')
print('Step 7: AI-Powered Verification & Cross-Source Analysis is COMPLETE.')

---

## Step 7 Complete — Structured Data Schema for Streamlit (Step 10)

The output of `verify_article(text)` produces a clean JSON-serializable dictionary ready for Step 10:

```python
{
    'ml_classification': {
        'prediction': 'REAL',
        'confidence': 86.72,
        'confidence_type': 'decision_margin (uncalibrated)',
        'model_used': 'LinearSVC'
    },
    'explainability_summary': {
        'influential_features': [...],
        'suspicious_language': [...]
    },
    'verification_summary': {
        'overall_status': 'HIGHLY_SUPPORTED',
        'disagreement_detected': False,
        'disagreement_warning': '',
        'claims_count': 3,
        'sources_checked_count': 5,
        'tier1_official_count': 2,
        'tier2_established_count': 3
    },
    'claim_verifications': [
        {
            'claim': '...',
            'status': 'SUPPORTED',
            'supporting_sources': ['Reuters', 'Federal Reserve (Official)'],
            'summary': '...'
        }
    ],
    'sources_analysis': {...},
    'ai_assessment': '...'
}
```